# 01 NASA data collection

Fetch official Site01 and regional inputs, validate byte integrity and preserve raw files. Site04 is gated by the integrated run. Use this notebook from the extracted package.

In [ ]:
from pathlib import Path
import os, sys
root=Path(os.environ.get("LUNARSIGHT_ROOT", Path.cwd())).resolve()
if not (root/"src/pipeline.py").exists() and (root.parent/"src/pipeline.py").exists():
    root=root.parent
assert (root/"src/pipeline.py").exists(), "Set LUNARSIGHT_ROOT to the extracted lunarsight-data folder."
sys.path.insert(0,str(root/"src"))
for name in ("reports","manifests","processed","figures"): (root/name).mkdir(exist_ok=True)

In [ ]:
from acquire import environment, sources, inventory
from pipeline import select_sources
import pandas as pd
environment(root)
items=sources()
rec={x["id"]:dict(x,status="NOT_REQUESTED") for x in items}
rec.update(select_sources(root,items,{x["id"] for x in items if x["site"]!="Site04"},False))
inventory(root,list(rec.values()))
table=pd.read_csv(root/"reports/data_inventory.csv")
display(table[["id","status","bytes","sha256"]])
assert all(x["status"] in {"VERIFIED_BYTES","NOT_REQUESTED"} for x in rec.values())